# Robustness Checks: q160 Update / q300 Evaluation

This notebook summarizes the robustness runs used in the thesis.

Most checks use four 400-timestamp windows and seeds `123`, `456`, and `789`.
The pricing-grid check is kept as the narrower seed-123 diagnostic.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import gamma, roots_genlaguerre
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
plt.style.use("seaborn-v0_8-whitegrid")


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    return here.parent if here.name == "notebooks" else here


def project_relative(path: Path) -> str:
    path = Path(path).resolve()
    try:
        return str(path.relative_to(PROJECT_ROOT)).replace(chr(92), "/")
    except ValueError:
        return str(path)


PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
ROBUST_ROOT = OUTPUT_ROOT / "robustness_q160_eval300_expanded"
BASELINE_ROOT = OUTPUT_ROOT / "common_eval_q160_eval300"
NONOVERLAP_ROOT = OUTPUT_ROOT / "nonoverlap_q160_eval300"
OLD_ROBUST_ROOT = OUTPUT_ROOT / "robustness_q160_eval300"
ANALYSIS_DIR = ROBUST_ROOT / "_analysis" / "combined"
TABLE_DIR = ANALYSIS_DIR / "tables"
FIGURE_DIR = ANALYSIS_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [123, 456, 789]
GRID_SEED = 123

WINDOWS = [
    {"window": "0-399", "robust_window": "w0_400", "baseline_window": None},
    {"window": "400-799", "robust_window": "w400_800", "baseline_window": "w401_800"},
    {"window": "800-1199", "robust_window": "w800_1200", "baseline_window": "w801_1200"},
    {"window": "1200-1599", "robust_window": "w1200_1600", "baseline_window": "w1201_1600"},
]

BASELINE = {"H": 0.20, "L": 8, "N": 300, "mc_paths": 512, "eval_mc_paths": 1024}

if not ROBUST_ROOT.exists():
    raise FileNotFoundError(f"Missing expanded robustness folder: {ROBUST_ROOT}")

print("Project root:", project_relative(PROJECT_ROOT))
print("Expanded robustness folder:", project_relative(ROBUST_ROOT))

## Run Inventory

In [ ]:
def baseline_run_path(model: str, window_info: dict, seed: int) -> Path:
    if window_info["window"] == "0-399":
        if model == "Normal SABR":
            return BASELINE_ROOT / f"normal_400ts_q160_eval300_seed{seed}"
        return BASELINE_ROOT / f"rough_400ts_q160_eval300_seed{seed}"

    base = NONOVERLAP_ROOT / window_info["baseline_window"]
    if model == "Normal SABR":
        return base / f"normal_sabr_no_A_RW_seed{seed}"
    return base / f"rough_sabr_logU0_seed{seed}"


def expanded_run_path(check: str, model: str, window_info: dict, seed: int, **kwargs) -> Path:
    w = window_info["robust_window"]
    if check == "N500":
        folder = ROBUST_ROOT / "N500" / w
        if model == "Normal SABR":
            return folder / f"normal_sabr_N500_seed{seed}"
        return folder / f"rough_sabr_H020_L8_N500_seed{seed}"
    if check == "mc_budget":
        folder = ROBUST_ROOT / "mc_budget" / w
        if model == "Normal SABR":
            return folder / f"normal_sabr_N300_mc1024_eval2048_seed{seed}"
        return folder / f"rough_sabr_H020_L8_N300_mc1024_eval2048_seed{seed}"
    if check == "L":
        L = int(kwargs["L"])
        return ROBUST_ROOT / "L_sensitivity" / w / f"rough_sabr_H020_L{L}_N300_seed{seed}"
    if check == "H":
        H_code = int(round(100 * kwargs["H"]))
        return ROBUST_ROOT / "H_sensitivity" / w / f"rough_sabr_H{H_code:03d}_L8_N300_seed{seed}"
    raise ValueError(check)


def grid_run_path(model: str, grid: str, seed: int = GRID_SEED) -> Path:
    if grid == "2d / max80":
        if model == "Normal SABR":
            return OLD_ROBUST_ROOT / "baseline_N300_mc512_eval1024" / f"normal_sabr_seed{seed}"
        return OLD_ROBUST_ROOT / "baseline_H020_L8_N300_mc512_eval1024" / f"rough_sabr_seed{seed}"
    if grid == "1d / max160":
        if model == "Normal SABR":
            return ROBUST_ROOT / "mc_grid" / "w0_400" / f"normal_sabr_N300_grid1d_seed{seed}"
        return ROBUST_ROOT / "mc_grid" / "w0_400" / f"rough_sabr_H020_L8_N300_grid1d_seed{seed}"
    if grid == "0.5d / max240":
        if model == "Normal SABR":
            return ROBUST_ROOT / "mc_grid" / "w0_400" / f"normal_sabr_N300_grid05d_seed{seed}"
        return ROBUST_ROOT / "mc_grid" / "w0_400" / f"rough_sabr_H020_L8_N300_grid05d_seed{seed}"
    raise ValueError(grid)


def add_run(rows, check, setting, model, window_info, seed, path, **meta):
    row = {
        "check": check,
        "setting": setting,
        "model": model,
        "window": window_info["window"],
        "robust_window": window_info["robust_window"],
        "seed": seed,
        "path": path,
        "H": np.nan,
        "L": np.nan,
        "N": BASELINE["N"],
        "mc_paths": BASELINE["mc_paths"],
        "eval_mc_paths": BASELINE["eval_mc_paths"],
    }
    row.update(meta)
    rows.append(row)


inventory_rows = []
for w in WINDOWS:
    for seed in SEEDS:
        add_run(inventory_rows, "baseline", "baseline", "Normal SABR", w, seed, baseline_run_path("Normal SABR", w, seed), N=300)
        add_run(inventory_rows, "baseline", "baseline", "Rough-SABR", w, seed, baseline_run_path("Rough-SABR", w, seed), H=0.20, L=8, N=300)
        add_run(inventory_rows, "particles", "N=500", "Normal SABR", w, seed, expanded_run_path("N500", "Normal SABR", w, seed), N=500)
        add_run(inventory_rows, "particles", "N=500", "Rough-SABR", w, seed, expanded_run_path("N500", "Rough-SABR", w, seed), H=0.20, L=8, N=500)
        add_run(inventory_rows, "MC budget", "1024/2048", "Normal SABR", w, seed, expanded_run_path("mc_budget", "Normal SABR", w, seed), N=300, mc_paths=1024, eval_mc_paths=2048)
        add_run(inventory_rows, "MC budget", "1024/2048", "Rough-SABR", w, seed, expanded_run_path("mc_budget", "Rough-SABR", w, seed), H=0.20, L=8, N=300, mc_paths=1024, eval_mc_paths=2048)
        for L in [16, 24]:
            add_run(inventory_rows, "L", f"L={L}", "Rough-SABR", w, seed, expanded_run_path("L", "Rough-SABR", w, seed, L=L), H=0.20, L=L, N=300)
        for H in [0.10, 0.30]:
            add_run(inventory_rows, "H", f"H={H:.2f}", "Rough-SABR", w, seed, expanded_run_path("H", "Rough-SABR", w, seed, H=H), H=H, L=8, N=300)

run_inventory = pd.DataFrame(inventory_rows)
run_inventory["exists"] = run_inventory["path"].apply(lambda p: p.exists())
run_inventory["run_dir"] = run_inventory["path"].apply(project_relative)

inventory_view = (
    run_inventory.groupby(["check", "setting", "model"], dropna=False)
    .agg(
        expected_runs=("run_dir", "size"),
        available_runs=("exists", "sum"),
        windows=("window", lambda s: ", ".join(sorted(s.unique()))),
        seeds=("seed", lambda s: ", ".join(map(str, sorted(s.unique())))),
    )
    .reset_index()
)

missing_runs = run_inventory.loc[~run_inventory["exists"], ["check", "setting", "model", "window", "seed", "run_dir"]]
if len(missing_runs):
    display(missing_runs)
else:
    print("All expected robustness runs are available.")

display(inventory_view)

## Load Scores

In [ ]:
def read_first_row(path: Path) -> dict:
    if not path.exists():
        return {}
    return pd.read_csv(path).iloc[0].to_dict()


def load_run(row) -> tuple[dict, pd.DataFrame]:
    run_dir = Path(row["path"])
    eval_summary = read_first_row(run_dir / "model_comparison_summary_eval.csv")
    update_summary = read_first_row(run_dir / "model_comparison_summary.csv")

    pred_cols = [
        "capture_time_utc",
        "t_index",
        "n_quotes",
        "n_update_quotes",
        "log_predictive_likelihood",
        "avg_log_predictive_likelihood_per_quote",
        "log_predictive_likelihood_with_forward_anchor",
        "avg_log_predictive_likelihood_with_forward_anchor_per_quote",
        "eval_n_options",
        "eval_n_priced",
    ]
    pred_path = run_dir / "predictive_loglikelihood_eval.csv"
    pred = pd.read_csv(pred_path, usecols=lambda c: c in pred_cols)
    pred["invalid_pricings"] = pred["eval_n_options"] - pred["eval_n_priced"]

    out = row.drop(labels=["path", "exists"], errors="ignore").to_dict()
    out["run_dir"] = project_relative(run_dir)

    for key in [
        "n_timestamps",
        "n_quotes_total",
        "total_log_predictive_likelihood",
        "avg_log_predictive_likelihood_per_quote",
        "total_log_predictive_likelihood_with_forward_anchor",
        "avg_log_predictive_likelihood_with_forward_anchor_per_quote",
        "mean_timestamp_log_predictive_likelihood",
        "mean_ess",
        "min_ess",
        "total_runtime_seconds",
        "mean_timestamp_seconds",
        "total_predictive_eval_seconds",
    ]:
        out[key] = eval_summary.get(key, np.nan)

    for key in [
        "mean_fit_price_rmse",
        "mean_fit_mean_abs_price_residual",
        "mean_fit_inside_bidask_rate",
    ]:
        out[key] = update_summary.get(key, np.nan)

    runtime_path = run_dir / "runtime_by_timestamp.csv"
    runtime = pd.read_csv(runtime_path) if runtime_path.exists() else pd.DataFrame()
    if len(runtime):
        out["runtime_minutes"] = runtime["timestamp_seconds"].sum() / 60
        out["predictive_eval_minutes"] = runtime.get("predictive_eval_seconds", pd.Series(dtype=float)).sum() / 60
    else:
        out["runtime_minutes"] = out["total_runtime_seconds"] / 60
        out["predictive_eval_minutes"] = out["total_predictive_eval_seconds"] / 60

    ess_path = run_dir / "ess.csv"
    ess = pd.read_csv(ess_path) if ess_path.exists() else pd.DataFrame()
    if len(ess):
        out["mean_ess_ratio"] = ess["ess"].mean() / out["N"]
        out["min_ess_ratio"] = ess["ess"].min() / out["N"]
        out["mean_tempering_steps"] = ess["n_tempering_steps"].mean()
    else:
        out["mean_ess_ratio"] = np.nan
        out["min_ess_ratio"] = np.nan
        out["mean_tempering_steps"] = np.nan

    for key, value in out.items():
        pred[key] = value
    return out, pred


records = []
timestamp_frames = []
for _, row in run_inventory[run_inventory["exists"]].iterrows():
    record, frame = load_run(row)
    records.append(record)
    timestamp_frames.append(frame)

run_summary = pd.DataFrame(records)
timestamp_scores = pd.concat(timestamp_frames, ignore_index=True)

run_summary.to_csv(TABLE_DIR / "expanded_robustness_run_summary.csv", index=False)
timestamp_scores.to_csv(TABLE_DIR / "expanded_robustness_timestamp_scores.csv", index=False)

print(f"Loaded {len(run_summary)} runs and {len(timestamp_scores):,} timestamp scores.")
display(run_summary.groupby(["check", "setting", "model"]).size().rename("runs").reset_index())

## Summary Helpers

In [ ]:
def aggregate_runs(data: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    grouped = data.groupby(group_cols, dropna=False)
    out = grouped.agg(
        runs=("run_dir", "size"),
        windows=("window", "nunique"),
        seeds=("seed", "nunique"),
        timestamps=("n_timestamps", "sum"),
        q300_quotes=("n_quotes_total", "sum"),
        total_score=("total_log_predictive_likelihood", "sum"),
        mean_timestamp_score=("mean_timestamp_log_predictive_likelihood", "mean"),
        mean_ess_ratio=("mean_ess_ratio", "mean"),
        min_ess_ratio=("min_ess_ratio", "min"),
        mean_tempering_steps=("mean_tempering_steps", "mean"),
    ).reset_index()
    out["score_per_quote"] = out["total_score"] / out["q300_quotes"]

    bench_cols = group_cols + ["runtime_minutes"]
    bench = data[(data["window"].eq("0-399")) & (data["seed"].eq(123))][bench_cols].copy()
    bench = bench.rename(columns={"runtime_minutes": "runtime_min_seed123_w0"})
    out = out.merge(bench, on=group_cols, how="left")
    return out


variant_cols = ["check", "setting", "model", "H", "L", "N", "mc_paths", "eval_mc_paths"]
variant_summary = aggregate_runs(run_summary, variant_cols)
variant_summary.to_csv(TABLE_DIR / "expanded_robustness_variant_summary.csv", index=False)

baseline_by_model = variant_summary[variant_summary["check"].eq("baseline")][
    ["model", "score_per_quote", "runtime_min_seed123_w0"]
].rename(
    columns={
        "score_per_quote": "model_baseline_score_per_quote",
        "runtime_min_seed123_w0": "model_baseline_runtime_min_seed123_w0",
    }
)
variant_summary = variant_summary.merge(baseline_by_model, on="model", how="left")
variant_summary["score_change_vs_model_baseline"] = (
    variant_summary["score_per_quote"] - variant_summary["model_baseline_score_per_quote"]
)
variant_summary["runtime_multiple_vs_model_baseline"] = (
    variant_summary["runtime_min_seed123_w0"] / variant_summary["model_baseline_runtime_min_seed123_w0"]
)

view_cols = [
    "check",
    "setting",
    "model",
    "H",
    "L",
    "N",
    "mc_paths",
    "eval_mc_paths",
    "runs",
    "windows",
    "seeds",
    "score_per_quote",
    "score_change_vs_model_baseline",
    "runtime_min_seed123_w0",
    "runtime_multiple_vs_model_baseline",
]
display(variant_summary[view_cols].round({
    "H": 2,
    "score_per_quote": 6,
    "score_change_vs_model_baseline": 6,
    "runtime_min_seed123_w0": 2,
    "runtime_multiple_vs_model_baseline": 2,
}))

## Particle Count

In [ ]:
particle_table = variant_summary[
    variant_summary["check"].isin(["baseline", "particles"])
    & variant_summary["model"].isin(["Normal SABR", "Rough-SABR"])
].copy()
particle_table = particle_table.sort_values(["N", "model"])

particle_view = particle_table[[
    "N",
    "model",
    "runs",
    "windows",
    "seeds",
    "score_per_quote",
    "score_change_vs_model_baseline",
    "runtime_min_seed123_w0",
    "runtime_multiple_vs_model_baseline",
]].rename(columns={"model": "Model"})

particle_view.to_csv(TABLE_DIR / "particle_count_robustness_table.csv", index=False)
display(particle_view.round({
    "score_per_quote": 6,
    "score_change_vs_model_baseline": 6,
    "runtime_min_seed123_w0": 2,
    "runtime_multiple_vs_model_baseline": 2,
}))

## Rough-SABR: Lift Dimension and H

In [ ]:
rough_only = variant_summary[variant_summary["model"].eq("Rough-SABR")].copy()

lift_table = rough_only[rough_only["check"].isin(["baseline", "L"])].sort_values("L").copy()
base_lift_score = lift_table[lift_table["L"].eq(8)]["score_per_quote"].iloc[0]
base_lift_runtime = lift_table[lift_table["L"].eq(8)]["runtime_min_seed123_w0"].iloc[0]
lift_table["score_change_vs_L8"] = lift_table["score_per_quote"] - base_lift_score
lift_table["runtime_multiple_vs_L8"] = lift_table["runtime_min_seed123_w0"] / base_lift_runtime

lift_view = lift_table[[
    "L",
    "runs",
    "windows",
    "seeds",
    "score_per_quote",
    "score_change_vs_L8",
    "runtime_min_seed123_w0",
    "runtime_multiple_vs_L8",
]]
lift_view.to_csv(TABLE_DIR / "lift_dimension_robustness_table.csv", index=False)
display(lift_view.round({
    "score_per_quote": 6,
    "score_change_vs_L8": 6,
    "runtime_min_seed123_w0": 2,
    "runtime_multiple_vs_L8": 2,
}))

hurst_table = rough_only[rough_only["check"].isin(["baseline", "H"])].sort_values("H").copy()
base_h_score = hurst_table[hurst_table["H"].round(2).eq(0.20)]["score_per_quote"].iloc[0]
hurst_table["score_change_vs_H020"] = hurst_table["score_per_quote"] - base_h_score

hurst_view = hurst_table[[
    "H",
    "runs",
    "windows",
    "seeds",
    "score_per_quote",
    "score_change_vs_H020",
    "runtime_min_seed123_w0",
]]
hurst_view.to_csv(TABLE_DIR / "hurst_robustness_table.csv", index=False)
display(hurst_view.round({
    "H": 2,
    "score_per_quote": 6,
    "score_change_vs_H020": 6,
    "runtime_min_seed123_w0": 2,
}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(lift_table["L"], lift_table["score_per_quote"], marker="o", color="tab:green")
axes[0].axvline(8, color="0.25", linestyle="--", linewidth=1)
axes[0].set_title("Lift dimension")
axes[0].set_xlabel("L")
axes[0].set_ylabel("Score per quote")

axes[1].plot(hurst_table["H"], hurst_table["score_per_quote"], marker="o", color="tab:purple")
axes[1].axvline(0.20, color="0.25", linestyle="--", linewidth=1)
axes[1].set_title("Hurst parameter")
axes[1].set_xlabel("H")
axes[1].set_ylabel("Score per quote")

fig.tight_layout()
fig.savefig(FIGURE_DIR / "rough_lift_and_h_score_robustness.png", dpi=180, bbox_inches="tight")
plt.show()

## H Change by Window

In [ ]:
h_hac_path = ROBUST_ROOT / "_analysis" / "h_sensitivity" / "tables" / "h_robustness_change_vs_H020_hac.csv"
if h_hac_path.exists():
    h_change_by_window = pd.read_csv(h_hac_path)
else:
    h_scores = timestamp_scores[timestamp_scores["model"].eq("Rough-SABR") & timestamp_scores["check"].isin(["baseline", "H"])].copy()
    h_scores["H_key"] = h_scores["H"].round(2)
    base_h = h_scores[h_scores["H_key"].eq(0.20)][["window", "seed", "t_index", "log_predictive_likelihood"]].rename(
        columns={"log_predictive_likelihood": "score_H020"}
    )
    alt_h = h_scores[~h_scores["H_key"].eq(0.20)].merge(base_h, on=["window", "seed", "t_index"], how="inner")
    alt_h["score_change_vs_H020"] = alt_h["log_predictive_likelihood"] - alt_h["score_H020"]
    h_change_by_window = (
        alt_h.groupby(["H", "window"])
        .agg(
            timestamps=("t_index", "nunique"),
            seeds=("seed", "nunique"),
            mean_score_change_vs_H020=("score_change_vs_H020", "mean"),
            median_score_change_vs_H020=("score_change_vs_H020", "median"),
            share_timestamps_above_H020=("score_change_vs_H020", lambda s: float((s > 0).mean())),
        )
        .reset_index()
    )

h_change_by_window.to_csv(TABLE_DIR / "hurst_change_vs_H020_by_window.csv", index=False)
display(h_change_by_window.round({
    "H": 2,
    "mean_score_change_vs_H020": 4,
    "median_score_change_vs_H020": 4,
    "share_timestamps_above_H020": 3,
    "hac_se": 4,
    "hac_ci_low": 4,
    "hac_ci_high": 4,
}))

## Kernel Error for H Values

In [ ]:
def kernel_error_table(H_values=(0.10, 0.20, 0.30), L=8):
    check_days = np.array([5 / (24 * 60), 1.0, 7.0, 30.0, 65.44])
    labels = ["5 min", "1 day", "7 days", "30 days", "65.44 days"]
    check_years = check_days / 365.0

    rows = []
    for H in H_values:
        alpha = -(H + 0.5)
        nodes, raw_weights = roots_genlaguerre(L, alpha)
        weights = raw_weights * np.exp(nodes) / gamma(0.5 - H)
        true_points = check_years ** (H - 0.5)
        lifted_points = np.sum(weights[:, None] * np.exp(-nodes[:, None] * check_years[None, :]), axis=0)
        errors = 100 * np.abs(lifted_points - true_points) / true_points
        row = {"H": H, "L": L}
        for label, error in zip(labels, errors):
            row[label] = error
        rows.append(row)
    return pd.DataFrame(rows)


h_kernel_errors = kernel_error_table()
h_kernel_errors.to_csv(TABLE_DIR / "hurst_kernel_error_L8.csv", index=False)
display(h_kernel_errors.round({"H": 2, "5 min": 2, "1 day": 2, "7 days": 2, "30 days": 2, "65.44 days": 4}))

## Matched Rough-Minus-Normal Checks

In [ ]:
def pair_model_scores(check: str, setting: str | None = None) -> pd.DataFrame:
    data = timestamp_scores[timestamp_scores["check"].eq(check)].copy()
    if setting is not None:
        data = data[data["setting"].eq(setting)].copy()
    rough = data[data["model"].eq("Rough-SABR")]
    normal = data[data["model"].eq("Normal SABR")]
    keys = ["window", "seed", "t_index", "N", "mc_paths", "eval_mc_paths"]
    paired = rough.merge(normal, on=keys, suffixes=("_rough", "_normal"))
    paired["score_diff"] = paired["log_predictive_likelihood_rough"] - paired["log_predictive_likelihood_normal"]
    paired["score_diff_per_quote"] = (
        paired["avg_log_predictive_likelihood_per_quote_rough"]
        - paired["avg_log_predictive_likelihood_per_quote_normal"]
    )
    paired["rough_wins"] = paired["score_diff"] > 0
    paired["check"] = check
    paired["setting"] = paired["setting_rough"]
    return paired


paired_baseline = pair_model_scores("baseline")
paired_n500 = pair_model_scores("particles", "N=500")
paired_high_mc = pair_model_scores("MC budget", "1024/2048")
paired_scores = pd.concat([paired_baseline, paired_n500, paired_high_mc], ignore_index=True)

paired_summary = (
    paired_scores.groupby(["check", "setting", "window"])
    .agg(
        paired_rows=("rough_wins", "size"),
        timestamps=("t_index", "nunique"),
        seeds=("seed", "nunique"),
        mean_score_diff=("score_diff", "mean"),
        cumulative_score_diff=("score_diff", "sum"),
        mean_score_diff_per_quote=("score_diff_per_quote", "mean"),
        rough_wins=("rough_wins", "sum"),
    )
    .reset_index()
)
paired_summary["rough_win_rate"] = paired_summary["rough_wins"] / paired_summary["paired_rows"]

overall = (
    paired_scores.groupby(["check", "setting"])
    .agg(
        paired_rows=("rough_wins", "size"),
        timestamps=("t_index", "nunique"),
        seeds=("seed", "nunique"),
        mean_score_diff=("score_diff", "mean"),
        cumulative_score_diff=("score_diff", "sum"),
        mean_score_diff_per_quote=("score_diff_per_quote", "mean"),
        rough_wins=("rough_wins", "sum"),
    )
    .reset_index()
)
overall["window"] = "Overall"
overall["rough_win_rate"] = overall["rough_wins"] / overall["paired_rows"]

paired_report = pd.concat([paired_summary, overall], ignore_index=True)
paired_report.to_csv(TABLE_DIR / "paired_rough_minus_normal_score_summary.csv", index=False)
display(paired_report.round({
    "mean_score_diff": 4,
    "cumulative_score_diff": 2,
    "mean_score_diff_per_quote": 6,
    "rough_win_rate": 3,
}))

In [ ]:
mc_compare = paired_report[paired_report["check"].isin(["baseline", "MC budget"])].copy()
mc_pivot = mc_compare.pivot_table(
    index="window",
    columns="check",
    values="mean_score_diff_per_quote",
    aggfunc="first",
).reset_index()
mc_pivot["change_high_mc_minus_baseline"] = mc_pivot["MC budget"] - mc_pivot["baseline"]
mc_pivot = mc_pivot.sort_values("window", key=lambda s: s.map({"0-399": 0, "400-799": 1, "800-1199": 2, "1200-1599": 3, "Overall": 4}))
mc_pivot.to_csv(TABLE_DIR / "mc_budget_score_difference_table.csv", index=False)
display(mc_pivot.round(6))

In [ ]:
plot_data = paired_report[paired_report["window"].ne("Overall")].copy()
plot_data["label"] = plot_data["check"].replace({"baseline": "baseline", "particles": "N=500", "MC budget": "MC 1024/2048"})

fig, ax = plt.subplots(figsize=(9, 4.5))
for label, group in plot_data.groupby("label"):
    group = group.sort_values("window", key=lambda s: s.map({"0-399": 0, "400-799": 1, "800-1199": 2, "1200-1599": 3}))
    ax.plot(group["window"], group["mean_score_diff_per_quote"], marker="o", label=label)

ax.axhline(0, color="0.25", linestyle="--", linewidth=1)
ax.set_title("Rough-minus-Normal score difference")
ax.set_xlabel("Window")
ax.set_ylabel("Mean score difference per quote")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "paired_score_difference_by_window.png", dpi=180, bbox_inches="tight")
plt.show()

## Pricing Grid

In [ ]:
GRID_RUNS = [
    {"model": "Normal SABR", "grid": "2d / max80", "grid_order": 0, "grid_days": 2.0},
    {"model": "Rough-SABR", "grid": "2d / max80", "grid_order": 0, "grid_days": 2.0},
    {"model": "Normal SABR", "grid": "1d / max160", "grid_order": 1, "grid_days": 1.0},
    {"model": "Rough-SABR", "grid": "1d / max160", "grid_order": 1, "grid_days": 1.0},
    {"model": "Normal SABR", "grid": "0.5d / max240", "grid_order": 2, "grid_days": 0.5},
    {"model": "Rough-SABR", "grid": "0.5d / max240", "grid_order": 2, "grid_days": 0.5},
]


def grid_price_stats(run_dir: Path) -> dict:
    prices = pd.read_csv(
        run_dir / "predictive_option_prices_eval.csv",
        usecols=["t_index", "predictive_price_error_vs_market", "predictive_mean_inside_bidask"],
    )
    prices["abs_error"] = prices["predictive_price_error_vs_market"].abs()
    prices["sq_error"] = prices["predictive_price_error_vs_market"] ** 2
    by_t = (
        prices.groupby("t_index")
        .agg(
            mae=("abs_error", "mean"),
            rmse=("sq_error", lambda x: float(np.sqrt(np.mean(x)))),
            inside_bidask=("predictive_mean_inside_bidask", "mean"),
        )
        .reset_index()
    )
    return {
        "MAE": by_t["mae"].mean(),
        "RMSE": by_t["rmse"].mean(),
        "inside_bidask": by_t["inside_bidask"].mean(),
    }


grid_rows = []
for run in GRID_RUNS:
    run_dir = grid_run_path(run["model"], run["grid"], GRID_SEED)
    summary = read_first_row(run_dir / "model_comparison_summary_eval.csv")
    runtime = pd.read_csv(run_dir / "runtime_by_timestamp.csv")
    row = dict(run)
    row["seed"] = GRID_SEED
    row["run_dir"] = project_relative(run_dir)
    row["score_per_quote"] = summary["avg_log_predictive_likelihood_per_quote"]
    row["mean_timestamp_score"] = summary["mean_timestamp_log_predictive_likelihood"]
    row["runtime_minutes"] = runtime["timestamp_seconds"].sum() / 60
    row.update(grid_price_stats(run_dir))
    grid_rows.append(row)

grid_summary = pd.DataFrame(grid_rows).sort_values(["grid_order", "model"]).reset_index(drop=True)
grid_summary.to_csv(TABLE_DIR / "pricing_grid_run_summary.csv", index=False)

grid_pairs = []
for grid, group in grid_summary.groupby("grid", sort=False):
    normal = group[group["model"].eq("Normal SABR")].iloc[0]
    rough = group[group["model"].eq("Rough-SABR")].iloc[0]
    grid_pairs.append({
        "Pricing grid": grid,
        "grid_order": int(rough["grid_order"]),
        "Normal score/q": normal["score_per_quote"],
        "Rough score/q": rough["score_per_quote"],
        "Score diff": rough["score_per_quote"] - normal["score_per_quote"],
        "Normal RMSE": normal["RMSE"],
        "Rough RMSE": rough["RMSE"],
        "RMSE diff": rough["RMSE"] - normal["RMSE"],
        "Runtime min": rough["runtime_minutes"] + normal["runtime_minutes"],
    })

grid_table = pd.DataFrame(grid_pairs).sort_values("grid_order").reset_index(drop=True)
grid_table["Runtime mult."] = grid_table["Runtime min"] / grid_table.loc[0, "Runtime min"]
grid_table = grid_table.drop(columns="grid_order")
grid_table.to_csv(TABLE_DIR / "pricing_grid_diagnostic_table.csv", index=False)
display(grid_table.round({
    "Normal score/q": 6,
    "Rough score/q": 6,
    "Score diff": 6,
    "Normal RMSE": 6,
    "Rough RMSE": 6,
    "RMSE diff": 6,
    "Runtime min": 2,
    "Runtime mult.": 2,
}))

## Optional Full q300 Price Errors

In [ ]:
LOAD_FULL_Q300_PRICE_ERRORS = False

if LOAD_FULL_Q300_PRICE_ERRORS:
    price_rows = []
    for _, row in run_inventory[run_inventory["exists"]].iterrows():
        run_dir = Path(row["path"])
        prices = pd.read_csv(
            run_dir / "predictive_option_prices_eval.csv",
            usecols=["t_index", "predictive_price_error_vs_market", "predictive_mean_inside_bidask"],
        )
        prices["abs_error"] = prices["predictive_price_error_vs_market"].abs()
        prices["sq_error"] = prices["predictive_price_error_vs_market"] ** 2
        by_t = (
            prices.groupby("t_index")
            .agg(
                MAE=("abs_error", "mean"),
                RMSE=("sq_error", lambda x: float(np.sqrt(np.mean(x)))),
                inside_bidask=("predictive_mean_inside_bidask", "mean"),
            )
            .reset_index()
        )
        price_rows.append({
            "check": row["check"],
            "setting": row["setting"],
            "model": row["model"],
            "window": row["window"],
            "seed": row["seed"],
            "MAE": by_t["MAE"].mean(),
            "RMSE": by_t["RMSE"].mean(),
            "inside_bidask": by_t["inside_bidask"].mean(),
        })
    full_price_errors = pd.DataFrame(price_rows)
    full_price_errors.to_csv(TABLE_DIR / "expanded_robustness_q300_price_errors.csv", index=False)
    display(full_price_errors.head())
else:
    print("Skipped full q300 price-error scan. Set LOAD_FULL_Q300_PRICE_ERRORS=True to compute it.")

## Notes

The main robustness checks above use the expanded three-seed results.
The pricing-grid table is the narrow seed-123 discretisation check.